# human fMRI post analysis

这个 notebook 用来直接读取 `TrippleN/customize/human_fMRI/` 下已经计算好的结果（decoding / encoding / rsa），并做快速预览与汇总展示。


In [1]:
from __future__ import annotations

import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/media/ubuntu/sda/TrippleN")
RESULT_DIR = ROOT / "customize" / "human_fMRI"

assert RESULT_DIR.exists(), f"Not found: {RESULT_DIR}"
print("RESULT_DIR:", RESULT_DIR)
print("subfolders:", [p.name for p in RESULT_DIR.iterdir() if p.is_dir()])


RESULT_DIR: /media/ubuntu/sda/TrippleN/customize/human_fMRI
subfolders: ['rsa', 'selectivity', 'decoding', 'encoding']


In [2]:
_MODULE_RENAMES = {
    "numpy._core.numeric": "numpy.core.numeric",
    "numpy._core.multiarray": "numpy.core.multiarray",
    "numpy._core": "numpy.core",
}


class _CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        module = _MODULE_RENAMES.get(module, module)
        return super().find_class(module, name)


def load_pickle(path: Path):
    with path.open("rb") as f:
        try:
            return pickle.load(f)
        except ModuleNotFoundError:
            f.seek(0)
            return _CompatUnpickler(f).load()


def summarize_obj(obj, max_items: int = 10):
    t = type(obj)
    if isinstance(obj, dict):
        keys = list(obj.keys())
        preview = keys[:max_items]
        return {
            "type": t.__name__,
            "len": len(obj),
            "keys_preview": preview,
            "values_types_preview": [type(obj[k]).__name__ for k in preview],
        }
    if isinstance(obj, (list, tuple)):
        preview = list(obj[:max_items]) if len(obj) else []
        return {
            "type": t.__name__,
            "len": len(obj),
            "items_types_preview": [type(x).__name__ for x in preview],
        }
    if isinstance(obj, np.ndarray):
        return {
            "type": "ndarray",
            "shape": obj.shape,
            "dtype": str(obj.dtype),
            "min": float(np.nanmin(obj)) if obj.size else None,
            "max": float(np.nanmax(obj)) if obj.size else None,
        }
    if isinstance(obj, pd.DataFrame):
        return {
            "type": "DataFrame",
            "shape": obj.shape,
            "columns_preview": list(obj.columns[:max_items]),
        }
    if isinstance(obj, pd.Series):
        return {
            "type": "Series",
            "shape": obj.shape,
            "name": obj.name,
        }
    return {"type": t.__name__}


In [3]:
def list_pkls(folder: Path):
    if not folder.exists():
        return []
    return sorted(folder.glob("*.pkl"))


def load_folder(name: str):
    folder = RESULT_DIR / name
    files = list_pkls(folder)
    rows = []
    loaded = {}
    for p in files:
        obj = load_pickle(p)
        loaded[p.name] = obj
        info = summarize_obj(obj)
        rows.append({"module": name, "file": p.name, **info})

    df = pd.DataFrame(rows)
    return loaded, df


loaded_decoding, df_decoding = load_folder("decoding")
loaded_encoding, df_encoding = load_folder("encoding")
loaded_rsa, df_rsa = load_folder("rsa")

pd.concat([df_decoding, df_encoding, df_rsa], ignore_index=True)


,module,file,type,len,keys_preview,values_types_preview,shape,columns_preview
0,decoding,fmri_decoding_all_summary.pkl,dict,7.0,"[early, midventral, midlateral, midparietal, v...","[dict, dict, dict, dict, dict, dict, dict]",NaN,NaN
1,decoding,fmri_decoding_early.pkl,dict,11.0,"[alexnet_fc6, clip_vit_l14_image, clip_vit_l14...","[dict, dict, dict, dict, dict, dict, dict, dic...",NaN,NaN
2,decoding,fmri_decoding_early_detail.pkl,dict,11.0,"[alexnet_fc6, clip_vit_l14_image, clip_vit_l14...","[dict, dict, dict, dict, dict, dict, dict, dic...",NaN,NaN
3,decoding,fmri_decoding_lateral.pkl,dict,11.0,"[alexnet_fc6, clip_vit_l14_image, clip_vit_l14...","[dict, dict, dict, dict, dict, dict, dict, dic...",NaN,NaN
4,decoding,fmri_decoding_lateral_detail.pkl,dict,11.0,"[alexnet_fc6, clip_vit_l14_image, clip_vit_l14...","[dict, dict, dict, dict, dict, dict, dict, dic...",NaN,NaN
...,...,...,...,...,...,...,...,...
104,rsa,fmri_rsa_midparietal.pkl,dict,6.0,"[region_name, n_voxels, n_repeats, sample_size...","[str, int, int, int, list, list]",NaN,NaN
105,rsa,fmri_rsa_midventral.pkl,dict,6.0,"[region_name, n_voxels, n_repeats, sample_size...","[str, int, int, int, list, list]",NaN,NaN
106,rsa,fmri_rsa_parietal.pkl,dict,6.0,"[region_name, n_voxels, n_repeats, sample_size...","[str, int, int, int, list, list]",NaN,NaN
107,rsa,fmri_rsa_summary.pkl,dict,2.0,"[region_results, model_names]","[list, list]",NaN,NaN


In [4]:
def show_summary(obj, title: str):
    print("\n==", title, "==")
    info = summarize_obj(obj)
    print("summary:", info)
    if isinstance(obj, dict):
        keys = list(obj.keys())
        for k in keys[:10]:
            v = obj[k]
            print(f"- {k}: {type(v).__name__}")
            if isinstance(v, pd.DataFrame):
                display(v.head())
            elif isinstance(v, np.ndarray):
                print("  shape:", v.shape, "dtype:", v.dtype)
            elif isinstance(v, (list, tuple)):
                print("  len:", len(v))
            elif isinstance(v, (int, float, str)):
                print("  value:", v)


for name, loaded in [
    ("decoding", loaded_decoding),
    ("encoding", loaded_encoding),
    ("rsa", loaded_rsa),
]:
    if f"fmri_{name}_all_summary.pkl" in loaded:
        show_summary(loaded[f"fmri_{name}_all_summary.pkl"], f"{name}: fmri_{name}_all_summary.pkl")
    elif f"fmri_{name}_summary.pkl" in loaded:
        show_summary(loaded[f"fmri_{name}_summary.pkl"], f"{name}: fmri_{name}_summary.pkl")



== decoding: fmri_decoding_all_summary.pkl ==
summary: {'type': 'dict', 'len': 7, 'keys_preview': ['early', 'midventral', 'midlateral', 'midparietal', 'ventral', 'lateral', 'parietal'], 'values_types_preview': ['dict', 'dict', 'dict', 'dict', 'dict', 'dict', 'dict']}
- early: dict
- midventral: dict
- midlateral: dict
- midparietal: dict
- ventral: dict
- lateral: dict
- parietal: dict

== encoding: fmri_encoding_all_summary.pkl ==
summary: {'type': 'dict', 'len': 7, 'keys_preview': ['early', 'midventral', 'midlateral', 'midparietal', 'ventral', 'lateral', 'parietal'], 'values_types_preview': ['dict', 'dict', 'dict', 'dict', 'dict', 'dict', 'dict']}
- early: dict
- midventral: dict
- midlateral: dict
- midparietal: dict
- ventral: dict
- lateral: dict
- parietal: dict

== rsa: fmri_rsa_summary.pkl ==
summary: {'type': 'dict', 'len': 2, 'keys_preview': ['region_results', 'model_names'], 'values_types_preview': ['list', 'list']}
- region_results: list
  len: 7
- model_names: list
  len:

In [5]:
MODEL_NAMES = [
    "alexnet_fc6",
    "clip_vit_l14_image",
    "clip_vit_l14_text",
    "clip_rn50_image",
    "clip_rn50_text",
    "clip_rn101_image",
    "clip_rn101_text",
    "all_mpnet_base_v2",
    "dinov3_vitl16",
    "dinov3_convnext_base",
    "dinov3_vitb16",
]
REGION_NAMES = ["early", "midventral", "midlateral", "midparietal", "ventral", "lateral", "parietal"]

ENC_DIR = RESULT_DIR / "encoding"
assert ENC_DIR.exists(), f"Not found: {ENC_DIR}"

print("ENC_DIR:", ENC_DIR)
print("models:", len(MODEL_NAMES))
print("regions:", len(REGION_NAMES))


ENC_DIR: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding
models: 11
regions: 7


In [6]:
def load_encoding_df(region: str, model: str) -> pd.DataFrame:
    p = ENC_DIR / f"fmri_encoding_{region}_{model}.pkl"
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = load_pickle(p)
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"Expected DataFrame in {p.name}, got {type(obj).__name__}")
    if "normalized_correlation" not in obj.columns:
        raise KeyError(f"normalized_correlation not found in {p.name}. columns={list(obj.columns)}")
    return obj


def make_voxel_key(region: str, voxel_index) -> str:
    return f"{region}::{voxel_index}"


In [7]:
rows = []
voxel_rows = []

for region in REGION_NAMES:
    base_df = load_encoding_df(region, MODEL_NAMES[0])
    voxel_index_list = list(base_df.index)

    for voxel_idx in voxel_index_list:
        voxel_id = make_voxel_key(region, voxel_idx)
        voxel_rows.append({"voxel_id": voxel_id, "region": region, "voxel_index": voxel_idx})

    for model in MODEL_NAMES:
        df = load_encoding_df(region, model)
        if list(df.index) != voxel_index_list:
            df = df.reindex(voxel_index_list)
        s = df["normalized_correlation"].rename(model)
        for voxel_idx, val in s.items():
            rows.append({"voxel_id": make_voxel_key(region, voxel_idx), "model": model, "normalized_correlation": float(val) if pd.notna(val) else np.nan})

long_df = pd.DataFrame(rows)

norm_matrix = long_df.pivot(index="voxel_id", columns="model", values="normalized_correlation").reindex(columns=MODEL_NAMES)
voxel_info = pd.DataFrame(voxel_rows).drop_duplicates(subset=["voxel_id"]).set_index("voxel_id")

print("norm_matrix shape:", norm_matrix.shape)
print("voxel_info shape:", voxel_info.shape)
print("matrix columns match models:", list(norm_matrix.columns) == MODEL_NAMES)


norm_matrix shape: (27638, 11)
voxel_info shape: (27638, 2)
matrix columns match models: True


In [8]:
OUT_MATRIX = ENC_DIR / "fmri_encoding_normalized_correlation_matrix.pkl"
OUT_VOXEL_INFO = ENC_DIR / "fmri_encoding_voxel_info.csv"

norm_matrix.to_pickle(OUT_MATRIX)
voxel_info.to_csv(OUT_VOXEL_INFO)

print("saved:")
print("-", OUT_MATRIX)
print("-", OUT_VOXEL_INFO)


saved:
- /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/fmri_encoding_normalized_correlation_matrix.pkl
- /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/fmri_encoding_voxel_info.csv


In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

sns.set_style("white")
plt.rcParams["axes.grid"] = False

PLOTS_DIR = ENC_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = PLOTS_DIR  # figure subfolders created below
MEAN_FIG_DIR = FIG_DIR / "mean_norm_corr"
SCATTER_FIG_DIR = FIG_DIR / "scatter_vs_alexnet"
SLOPE_BAR_FIG_DIR = FIG_DIR / "slope_bar_vs_alexnet"
SLOPE_BOX_FIG_DIR = FIG_DIR / "slope_boxplot"
for _d in [MEAN_FIG_DIR, SCATTER_FIG_DIR, SLOPE_BAR_FIG_DIR, SLOPE_BOX_FIG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

DATA_DIR = PLOTS_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ALEXNET_MODEL = "alexnet_fc6"
assert ALEXNET_MODEL in MODEL_NAMES

def valid_xy(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    m = np.isfinite(x) & np.isfinite(y) & (x >= 0) & (x <= 1) & (y >= 0) & (y <= 1)
    return x[m], y[m]


def slope_no_intercept(x: np.ndarray, y: np.ndarray):
    x, y = valid_xy(x, y)
    if x.size == 0:
        return np.nan, 0
    denom = np.sum(x**2)
    if denom == 0:
        return np.nan, int(x.size)
    return float(np.sum(x * y) / denom), int(x.size)

print("PLOTS_DIR:", PLOTS_DIR)


PLOTS_DIR: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots


In [10]:
def region_voxel_ids(region: str):
    return voxel_info.index[voxel_info["region"] == region]


def region_matrix(region: str) -> pd.DataFrame:
    idx = region_voxel_ids(region)
    return norm_matrix.loc[idx]


def region_long_df(region: str) -> pd.DataFrame:
    mat = region_matrix(region)
    df = mat.reset_index(names="voxel_id").melt(id_vars=["voxel_id"], var_name="model", value_name="normalized_correlation")
    df = df[np.isfinite(df["normalized_correlation"]) & (df["normalized_correlation"] >= 0) & (df["normalized_correlation"] <= 1)]
    df["region"] = region
    return df


In [16]:
# 1) mean_norm_corr 柱状图：每个脑区一个 PDF（单页）

summary_parts = []
for region in REGION_NAMES:
    df_r = region_long_df(region)

    df_mean = (
        df_r.groupby("model", as_index=False)["normalized_correlation"]
        .mean()
        .rename(columns={"normalized_correlation": "mean_norm_corr"})
    )
    order = df_mean.sort_values("mean_norm_corr", ascending=True)["model"].tolist()

    (DATA_DIR / "mean_norm_corr").mkdir(parents=True, exist_ok=True)
    df_mean.to_csv(DATA_DIR / "mean_norm_corr" / f"{region}_mean_norm_corr_barplot.csv", index=False)
    summary_parts.append(df_mean.assign(region=region))

    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    sns.barplot(
        data=df_mean,
        x="model",
        y="mean_norm_corr",
        order=order,
        ax=ax,
        color="#5E9FD1",
        width=0.6,
    )
    ax.grid(False)
    ax.set_title(f"{region} | mean(normalized_correlation)")
    ax.set_xlabel("model")
    ax.set_ylabel("mean_norm_corr")
    ax.set_ylim(0, 0.3)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()

    out_path = MEAN_FIG_DIR / f"{region}_mean_norm_corr_barplot.pdf"
    fig.savefig(out_path)
    plt.close(fig)

from matplotlib.patches import Rectangle

df_summary = pd.concat(summary_parts, ignore_index=True)
df_summary.to_csv(DATA_DIR / "mean_norm_corr" / "all_regions_mean_norm_corr_summary.csv", index=False)

region_bar_colors = dict(
    zip(
        REGION_NAMES,
        ["#edce4a", "#7db57e", "#5692cd", "#9d72a7", "#a1645f", "#e19469", "#de7a87"],
    )
)
model_order_summary = (
    df_summary.groupby("model", as_index=True)["mean_norm_corr"]
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)

n_r = len(REGION_NAMES)
dx = 0.102
bar_w = 0.098
group_step = n_r * dx + 0.32
n_m = len(model_order_summary)
fig_w = 10
fig_h = 5.0
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
for mi, model in enumerate(model_order_summary):
    x0 = mi * group_step
    x_offs = np.linspace(-(n_r - 1) * dx / 2, (n_r - 1) * dx / 2, n_r)
    for ri, region in enumerate(REGION_NAMES):
        val = float(
            df_summary.loc[
                (df_summary["model"] == model) & (df_summary["region"] == region),
                "mean_norm_corr",
            ].iloc[0]
        )
        ax.bar(x0 + x_offs[ri], val, width=bar_w, color=region_bar_colors[region], linewidth=0)
half_span = (n_r - 1) * dx / 2 + bar_w / 2
pad_x = 0.12
ax.set_xticks(np.arange(n_m) * group_step)
ax.set_xticklabels(model_order_summary, rotation=45, ha="right")
ax.set_xlabel("model")
ax.set_ylabel("mean_norm_corr")
ax.set_ylim(0, 0.3)
ax.set_xlim(-half_span - pad_x, (n_m - 1) * group_step + half_span + pad_x)
ax.grid(False)
ax.legend(
    handles=[Rectangle((0, 0), 1, 1, fc=region_bar_colors[r]) for r in REGION_NAMES],
    labels=REGION_NAMES,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=False,
)
plt.tight_layout()
summary_out = MEAN_FIG_DIR / "all_models_regions_mean_norm_corr_bar_grouped.pdf"
fig.savefig(summary_out, bbox_inches="tight", pad_inches=0.15)
plt.close(fig)

print("saved mean barplots to:", PLOTS_DIR)
print("saved summary:", summary_out)


saved mean barplots to: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots
saved summary: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots/mean_norm_corr/all_models_regions_mean_norm_corr_bar_grouped.pdf


In [12]:
# 2) 各脑区 model vs alexnet scatter：一个脑区一个 PDF，每页一个 model

for region in REGION_NAMES:
    mat = region_matrix(region)
    x0 = mat[ALEXNET_MODEL].values

    out_pdf = SCATTER_FIG_DIR / f"{region}_scatter_vs_{ALEXNET_MODEL}.pdf"
    with PdfPages(out_pdf) as pdf:
        for model in MODEL_NAMES:
            if model == ALEXNET_MODEL:
                continue

            y0 = mat[model].values
            x0_arr = np.asarray(x0).ravel()
            y0_arr = np.asarray(y0).ravel()
            m = np.isfinite(x0_arr) & np.isfinite(y0_arr) & (x0_arr >= 0) & (x0_arr <= 1) & (y0_arr >= 0) & (y0_arr <= 1)
            x = x0_arr[m]
            y = y0_arr[m]

            # save plotting data (one csv per page)
            (DATA_DIR / "scatter_vs_alexnet" / region).mkdir(parents=True, exist_ok=True)
            pd.DataFrame(
                {
                    "voxel_id": mat.index.to_numpy()[m],
                    ALEXNET_MODEL: x,
                    model: y,
                }
            ).to_csv(DATA_DIR / "scatter_vs_alexnet" / region / f"{region}_scatter_{model}_vs_{ALEXNET_MODEL}.csv", index=False)

            fig, ax = plt.subplots(figsize=(5, 5))
            ax.scatter(x, y, s=1, alpha=1, color="#5E9FD1")
            ax.grid(False)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_xlabel(ALEXNET_MODEL)
            ax.set_ylabel(model)
            ax.set_title(region)
            ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)

            a, n = slope_no_intercept(x0, y0)
            if np.isfinite(a):
                ax.plot([0, 1], [0, a], color="#EC6F7E", linewidth=1.5)
                ax.text(0.05, 0.95, f"slope={a:.3f}", transform=ax.transAxes, ha="left", va="top", color="#EC6F7E")

            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

print("saved scatter pdfs to:", PLOTS_DIR)


saved scatter pdfs to: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots


In [18]:
# 3) 各脑区 slope 柱状图：一个脑区一个 PDF（基于 bootstrap voxel sampling）

BOOT_N_REP = 100
BOOT_N_SAMPLE = 5000
rng = np.random.default_rng(42)

slope_rows = []

for region in REGION_NAMES:
    mat = region_matrix(region)
    x_all = mat[ALEXNET_MODEL].values
    n_vox = len(x_all)

    for model in MODEL_NAMES:
        if model == ALEXNET_MODEL:
            continue
        y_all = mat[model].values

        for rep in range(BOOT_N_REP):
            idx = rng.choice(n_vox, size=min(BOOT_N_SAMPLE, n_vox), replace=(n_vox < BOOT_N_SAMPLE))
            a, n = slope_no_intercept(x_all[idx], y_all[idx])
            slope_rows.append({"region": region, "model": model, "rep": rep, "slope": a, "n": n})

slope_boot = pd.DataFrame(slope_rows)

for region in REGION_NAMES:
    df_r = slope_boot[slope_boot["region"] == region].copy()
    order = df_r.groupby("model")["slope"].mean().sort_values(ascending=True).index.tolist()

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(
        data=df_r,
        x="model",
        y="slope",
        order=order,
        color="#5E9FD1",
        errorbar="se",
        width=0.6,
        capsize=0.3,
        err_kws={"linewidth": 1.5},
        ax=ax,
    )
    ax.grid(False)
    ax.set_title(f"{region} | slope vs {ALEXNET_MODEL} (no intercept)")
    ax.set_xlabel("model (sorted by mean)")
    ax.set_ylabel("slope")
    ax.set_ylim(0, 1.3)
    ax.axhline(1, color="black", linestyle="--", linewidth=1, zorder=0)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()

    out_path = SLOPE_BAR_FIG_DIR / f"{region}_slope_bar_vs_{ALEXNET_MODEL}.pdf"
    fig.savefig(out_path)
    plt.close(fig)

    # save plotting data (region summary for this barplot)
    (DATA_DIR / "slope_bar_vs_alexnet").mkdir(parents=True, exist_ok=True)
    df_sum = (
        df_r.groupby("model", as_index=False)
        .agg(mean_slope=("slope", "mean"), std_slope=("slope", "std"), n_rep=("rep", "nunique"))
    )
    df_sum["sem_slope"] = df_sum["std_slope"] / np.sqrt(df_sum["n_rep"].clip(lower=1))
    df_sum.to_csv(DATA_DIR / "slope_bar_vs_alexnet" / f"{region}_slope_bar_vs_{ALEXNET_MODEL}.csv", index=False)

from matplotlib.patches import Rectangle

df_slope_summary = (
    slope_boot.groupby(["region", "model"], as_index=False)["slope"]
    .mean()
    .rename(columns={"slope": "mean_slope"})
)
df_slope_summary.to_csv(
    DATA_DIR / "slope_bar_vs_alexnet" / f"all_regions_grouped_mean_slope_vs_{ALEXNET_MODEL}.csv",
    index=False,
)

region_bar_colors = dict(
    zip(
        REGION_NAMES,
        ["#edce4a", "#7db57e", "#5692cd", "#9d72a7", "#a1645f", "#e19469", "#de7a87"],
    )
)
model_order_slope = (
    df_slope_summary.groupby("model", as_index=True)["mean_slope"]
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)

n_r = len(REGION_NAMES)
dx = 0.102
bar_w = 0.098
group_step = n_r * dx + 0.32
n_m = len(model_order_slope)
fig_w = 10
fig_h = 5.0
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
for mi, model in enumerate(model_order_slope):
    x0 = mi * group_step
    x_offs = np.linspace(-(n_r - 1) * dx / 2, (n_r - 1) * dx / 2, n_r)
    for ri, region in enumerate(REGION_NAMES):
        val = float(
            df_slope_summary.loc[
                (df_slope_summary["model"] == model) & (df_slope_summary["region"] == region),
                "mean_slope",
            ].iloc[0]
        )
        ax.bar(x0 + x_offs[ri], val, width=bar_w, color=region_bar_colors[region], linewidth=0)
half_span = (n_r - 1) * dx / 2 + bar_w / 2
pad_x = 0.12
ax.set_xticks(np.arange(n_m) * group_step)
ax.set_xticklabels(model_order_slope, rotation=45, ha="right")
ax.set_xlabel("model")
ax.set_ylabel("slope")
ax.set_ylim(0, 1.3)
ax.axhline(1, color="black", linestyle="--", linewidth=1, zorder=0)
ax.set_xlim(-half_span - pad_x, (n_m - 1) * group_step + half_span + pad_x)
ax.grid(False)
ax.legend(
    handles=[Rectangle((0, 0), 1, 1, fc=region_bar_colors[r]) for r in REGION_NAMES],
    labels=REGION_NAMES,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=False,
)
plt.tight_layout()
slope_grouped_out = SLOPE_BAR_FIG_DIR / f"all_models_regions_slope_bar_grouped_vs_{ALEXNET_MODEL}.pdf"
fig.savefig(slope_grouped_out, bbox_inches="tight", pad_inches=0.15)
plt.close(fig)

out_csv = PLOTS_DIR / f"slope_bootstrap_vs_{ALEXNET_MODEL}.csv"
slope_boot.to_csv(out_csv, index=False)

# save plotting data for all-regions slope boxplot (same source)
region_sum_csv = DATA_DIR / f"all_regions_slope_bootstrap_summary_vs_{ALEXNET_MODEL}.csv"
(
    slope_boot.groupby(["region", "model"], as_index=False)
    .agg(mean_slope=("slope", "mean"), std_slope=("slope", "std"), n_rep=("rep", "nunique"))
    .assign(sem_slope=lambda d: d["std_slope"] / np.sqrt(d["n_rep"].clip(lower=1)))
    .to_csv(region_sum_csv, index=False)
)

print("saved slopes:", out_csv)
print("saved slope summaries:", region_sum_csv)
print("saved slope grouped:", slope_grouped_out)


saved slopes: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots/slope_bootstrap_vs_alexnet_fc6.csv
saved slope summaries: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots/data/all_regions_slope_bootstrap_summary_vs_alexnet_fc6.csv
saved slope grouped: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots/slope_bar_vs_alexnet/all_models_regions_slope_bar_grouped_vs_alexnet_fc6.pdf


In [14]:
# 4) 各脑区 slope boxplot：一个 PDF，每一行一个脑区

fig, ax = plt.subplots(figsize=(6, 4.5))

# 每个 region 一行：把该 region 下所有 model 的 slope(bootstrap) 合在一起
sns.boxplot(
    data=slope_boot,
    y="region",
    x="slope",
    order=REGION_NAMES,
    color="#5E9FD1",
    showfliers=False,
    linewidth=1,
    ax=ax,
)
ax.grid(False)
ax.set_title(f"slope distribution vs {ALEXNET_MODEL} (bootstrap over voxels & models)")
ax.set_xlabel("slope")
ax.set_ylabel("region")
plt.tight_layout()

out_path = SLOPE_BOX_FIG_DIR / f"all_regions_slope_boxplot_vs_{ALEXNET_MODEL}.pdf"
fig.savefig(out_path)
plt.close(fig)

print("saved:", out_path)


saved: /media/ubuntu/sda/TrippleN/customize/human_fMRI/encoding/plots/slope_boxplot/all_regions_slope_boxplot_vs_alexnet_fc6.pdf


### Decoding

对 `decoding/fmri_decoding_{region}_detail.pkl` 中每个脑区、每个模型，用与 `scripts/decoding_accuracy_curve.py` 相同的 LOOCV 预测与 `target_reduced` 计算准确率–候选集大小曲线；结果写入同目录 `fmri_decoding_accuracy_curves.pkl` 与长表 `fmri_decoding_accuracy_curves_long.csv`。将 `DEC_ACC_QUICK = True` 可改为小网格与 1000 次 trial 试跑。

In [ ]:
import sys

_scripts = ROOT / "scripts"
if str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))

from decoding_accuracy_curve import compute_accuracy_curve

DEC_DIR = RESULT_DIR / "decoding"
DEC_ACC_QUICK = False
if DEC_ACC_QUICK:
    acc_n_values = [2, 10, 20]
    acc_n_trials = 1000
else:
    acc_n_values = [2] + list(range(10, 1001, 10))
    acc_n_trials = 50000
acc_seed = 42
n_images_expect = 1000

fmri_decoding_accuracy = {}
rows_acc = []

for region in REGION_NAMES:
    p = DEC_DIR / f"fmri_decoding_{region}_detail.pkl"
    if not p.exists():
        print("skip (missing):", p)
        continue
    detail = load_pickle(p)
    fmri_decoding_accuracy[region] = {}
    for model_name, data in detail.items():
        if not isinstance(data, dict):
            continue
        if "target_reduced" not in data or "predictions" not in data:
            print("skip", region, model_name, "(no predictions / target_reduced)")
            continue
        pred = np.asarray(data["predictions"], dtype=np.float64)
        tgt = np.asarray(data["target_reduced"], dtype=np.float64)
        if pred.shape[0] != n_images_expect or tgt.shape[0] != n_images_expect:
            print("skip", region, model_name, "n_samples", pred.shape[0], tgt.shape[0])
            continue
        print("compute", region, model_name, "trials=", acc_n_trials, "n_grid", len(acc_n_values))
        acc = compute_accuracy_curve(pred, tgt, acc_n_values, acc_n_trials, acc_seed)
        fmri_decoding_accuracy[region][model_name] = acc
        for n, a in acc.items():
            rows_acc.append(
                {"region": region, "model": model_name, "n": n, "accuracy": a}
            )

dec_acc_payload = {
    "n_values": acc_n_values,
    "n_trials": acc_n_trials,
    "seed": acc_seed,
    "quick": DEC_ACC_QUICK,
    "accuracy_curves_by_region": fmri_decoding_accuracy,
}

dec_acc_pkl = DEC_DIR / "fmri_decoding_accuracy_curves.pkl"
with dec_acc_pkl.open("wb") as f:
    pickle.dump(dec_acc_payload, f)
print("saved:", dec_acc_pkl)

if rows_acc:
    df_dec_acc = pd.DataFrame(rows_acc)
    dec_acc_csv = DEC_DIR / "fmri_decoding_accuracy_curves_long.csv"
    df_dec_acc.to_csv(dec_acc_csv, index=False)
    print("saved:", dec_acc_csv, "rows:", len(df_dec_acc))
    preview_n = [2, 10, 100, 1000]
    sub = df_dec_acc[df_dec_acc["n"].isin([x for x in preview_n if x in df_dec_acc["n"].values])]
    display(sub.sort_values(["region", "model", "n"]).head(40))
else:
    print("no curves computed (missing detail pkls or empty)")